Libraries

In [1]:
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime
import requests
import re
import time
import utils as utils

Paises que participan en el mundial y su ranking FIFA

Ranking FIFA

In [2]:
if __name__ == "__main__":
    # Pega aquí la URL exacta que sacaste de la pestaña Network (F12)
    # Ejemplo: "https://inside.fifa.com/api/ranking-overview?locale=en&dateId=id_del_ranking"
    url_tu_api = "https://api.fifa.com/api/v3/fifarankings/rankings/live?gender=1&sportType=0&language=en" 
    
    df_ranking_fifa = utils.obtener_ranking_fifa_oficial(url_tu_api)
    
    if not df_ranking_fifa.empty:
        print("\n--- VISTA PREVIA DEL RANKING FIFA OBTENIDO ---")
        print(df_ranking_fifa.sample(5).to_string(index=False))

Conectando y procesando el JSON de la FIFA...
¡Éxito! Se importaron 211 selecciones desde el Ranking FIFA.

--- VISTA PREVIA DEL RANKING FIFA OBTENIDO ---
 Ranking_Actual  Seleccion  Puntos_FIFA Codigo_Pais
             72    Jamaica  1357.840144         JAM
             96    Belarus  1242.876029         BLR
            113 Mauritania  1176.680994         MTN
             15        USA  1688.534820         USA
             87   Bulgaria  1271.683325         BUL


Webscrapping a pagina Wikipedia

Son los paises que están en Mundial 2026

In [3]:
url_mundial = "https://en.wikipedia.org/wiki/2026_FIFA_World_Cup"
headers = {'User-Agent': 'Mozilla/5.0'}

try:
    respuesta = requests.get(url_mundial, headers=headers)
    tablas = pd.read_html(respuesta.text)
except Exception as e:
    print(f"Error al leer las tablas del Mundial: {e}")

patron = r'([A-Za-z\s]+?)(?:\[.*?\])?\s*\((\d+)\)'
confederaciones_ignorar = ['AFC', 'CAF', 'CONCACAF', 'CONMEBOL', 'OFC', 'UEFA']

datos_procesados = []

for columna in tablas[5].columns:
    texto_celda = str(tablas[5][columna].iloc[0])
    
    # Aplicamos la expresión regular para buscar todos los países y rankings en esa celda
    coincidencias = re.findall(patron, texto_celda)
    
    # Limpiamos y guardamos los datos encontrados
    
    for pais, ranking in coincidencias:
        pais_limpio = pais.strip()
        
        if pais_limpio not in confederaciones_ignorar:
            # Aquí obtenemos el código automáticamente del diccionario
            codigo = utils.obtener_codigo_pais(pais_limpio, df_ranking_fifa)
            
            datos_procesados.append({
                "Seleccion": pais_limpio,
                "Codigo_Pais": codigo,
                "Ranking_FIFA": int(ranking)
            })

# 4. Convertimos la lista de todos los países encontrados en un DataFrame final
ranking_teams = pd.DataFrame(datos_procesados)
ranking_teams.replace({'ao': 'Curacao'}, inplace=True)
print(ranking_teams.sample(5).to_string(index=False))
print(len(ranking_teams))

  Seleccion Codigo_Pais  Ranking_FIFA
  Argentina         ARG             1
      Qatar         QAT            56
     Sweden         SWE            38
   Paraguay         PAR            41
South Korea         KOR            25
48


Capturar Partidos Anteriores / Caso Selección Irán 

In [4]:
def games_iran(url, codigo_pais, df_ranking_fifa):    
    print(f"Extrayendo para Irán/Tabla Compleja: {codigo_pais}...")
    headers = {'User-Agent': 'Mozilla/5.0'}

    try:
        respuesta = requests.get(url, headers=headers)
        soup = BeautifulSoup(respuesta.text, 'html.parser')
    except Exception as e:
        print(f"Error: {e}")
        return pd.DataFrame()

    partidos = []

    ciudad_a_pais = {
        'Al-Ain': 'United Arab Emirates',
        'Al-Rayyan': 'Qatar',
        'Amman': 'Jordan',
        'Antalya': 'Turkey',
        'Arad': 'Bahrain',
        'Ashgabat': 'Turkmenistan',
        'Bishkek': 'Kyrgyzstan',
        'Doha': 'Qatar',
        'Dubai': 'United Arab Emirates',
        'Fooladshahr': 'Iran',
        'Hisar': 'Tajikistan',
        'Hong Kong': 'Hong Kong',
        'Inglewood': 'USA',
        'Kish': 'Iran',
        'Maria Enzersdorf': 'Austria',
        'Mashhad': 'Iran',
        'Plovdiv': 'Bulgaria',
        'Riffa': 'Bahrain',
        'Sankt Pölten': 'Austria',
        'Sarajevo': 'Bosnia and Herzegovina',
        'Seattle': 'USA',
        'Seoul': 'South Korea',
        'Sidon': 'Lebanon',
        'Tashkent': 'Uzbekistan',
        'Tehran': 'Iran',
        'Vientiane': 'Laos',
        'Volgograd': 'Russia'
    }

    fecha_limite = pd.to_datetime("2022-12-18")
    año_actual = "2022" 

    # Buscamos filas (tr)
    filas = soup.find_all(['h2', "h3" , 'a', 'div', 'p'])

    for fila in filas:
    # Detectamos el año en encabezados como antes
        if fila.name == 'div' and 'mw-heading' in fila.get('class', []):
            match_año = re.search(r'(20\d{2})', fila.get_text())
            if match_año: año_actual = match_año.group(1)
            continue
        
        if fila.name == 'p':
            texto_p = fila.get_text(strip=True)
            if re.match(r'^(20\d{2}|Friendly)', texto_p, re.IGNORECASE):
                qualif = texto_p
            continue
        
        if fila.name == 'div' and 'footballbox' in fila.get('class', []):

            # 2. Extraer datos usando los nombres de clase que identificamos en el HTML
            fecha = fila.find('div', class_='fdate').get_text(strip=True)
            local = fila.find('th', class_='fhome').get_text(strip=True)
            marcador_crudo = fila.find('th', class_='fscore').get_text(strip=True)
            visita = fila.find('th', class_='faway').get_text(strip=True)
            
            # El estadio está en un div con clase 'fright'
            # Usamos .get_text() y luego limpiamos el texto
            venue_info = fila.find('div', class_='fright')
            venue = venue_info.find('span', itemprop='name address').get_text(strip=True) #if venue_info else "N/A"
            city_url = venue_info.find('span', itemprop='name address').find_all('a')[1]['href']
            city_text = venue_info.find('span', itemprop='name address').find_all('a')[1].get_text(strip=True)
            url_completa = f"https://en.wikipedia.org{city_url}"
            country = ciudad_a_pais.get(city_text, "Desconocido")
            
            fecha_dt = pd.to_datetime(fecha)
            if fecha_dt <= fecha_limite: continue

            # Extraemos goles
            match_marcador = re.search(r'(\d+)\s*[–-]\s*(\d+)', marcador_crudo)
            
            if not match_marcador:
                continue
            
            goles_1 = int(match_marcador.group(1))
            goles_2 = int(match_marcador.group(2))
            
            if local.lower() == 'iran':
                opposite = visita
                goles_equipo = goles_1
                goles_rival = goles_2                  
            else:
                opposite = local 
                goles_equipo = goles_2
                goles_rival = goles_1                     
            
            local_a = local.lower()
            estadio_a = country.lower()
            visita_a = visita.lower()

            if local_a in estadio_a:
                if utils.obtener_codigo_pais(local, df_ranking_fifa) == codigo_pais:
                    condicion = "L"
                else:
                    condicion = "V"
            elif visita_a in estadio_a:  
                if utils.obtener_codigo_pais(visita, df_ranking_fifa) == codigo_pais:                      
                    condicion = "L"
                else:
                    condicion = "V"               
            else:
                condicion = "N" 
                
            penal_info = fila.find('a', title='Penalty shoot-out (association football)')
            
            if not penal_info:
                pen = False
            else:
                pen = True     
            
        # Guardamos (ajustando índices de elementos_fila según lo detectado)
            partidos.append({
                "Seleccion": codigo_pais,
                "Fecha": fecha_dt.strftime("%Y-%m-%d"),
                "Tipo_Partido": qualif, # Suele ser el torneo
                "Venue": venue + ", "+ country,
                "Condicion": condicion,
                "Oponente": opposite,
                "Goles_Equipo": goles_equipo,
                "Goles_Rival": goles_rival,
                "Resultado_Original": marcador_crudo,
                "Es_Penales": pen,
                "Es_Tiempo_Extra": "a.e.t." in marcador_crudo.lower(),
            })      
    return pd.DataFrame(partidos)          

Captura BBDD según código país y tipo pagina wikipedia

In [5]:
lst_1 = ['ARG', 'COL', 'BRA', 'ECU', 'PAR', 'HAI', 'NZL', 'SCO']
lst_2 = ['URU', 'AUS', 'BEL', 'IRQ', 'JPN', 'JOR', 'QAT', 'KSA', 'KOR', 'UZB',
        'ALG', 'CPV', 'COD', 'EGY', 'GHA', 'CIV', 'MAR', 'SEN', 'RSA', 'TUN',
        'CUW', 'MEX', 'PAN', 'AUT', 'BIH', 'CRO', 'CZE', 'ENG', 'FRA', 
        'GER', 'NOR', 'POR', 'ESP', 'SWE', 'SUI', 'TUR']
lst_3 = ['USA', 'CAN']
lst_4 = ['NED']
lst_5 = ['IRN']

In [6]:
df_games = pd.DataFrame()

for lst in lst_1:
    _df = utils.games_type01(utils.records_teams[lst], lst)
    df_games = pd.concat([df_games, _df])
    
for lst in lst_2:
    _df = utils.games_type02(utils.records_teams[lst], lst, df_ranking_fifa)
    df_games = pd.concat([df_games, _df])
    
for lst in lst_3:
    _df = utils.games_type03(utils.records_teams[lst], lst, df_ranking_fifa)
    df_games = pd.concat([df_games, _df])
    
for lst in lst_4:
    _df = utils.games_type04(utils.records_teams[lst], lst, df_ranking_fifa)
    df_games = pd.concat([df_games, _df])    
    
for lst in lst_5:
    _df = games_iran(utils.records_teams[lst], lst, df_ranking_fifa)
    df_games = pd.concat([df_games, _df])                

Iniciando extracción de datos para: ARG...
Iniciando extracción de datos para: COL...
Iniciando extracción de datos para: BRA...
Iniciando extracción de datos para: ECU...
Iniciando extracción de datos para: PAR...
Iniciando extracción de datos para: HAI...
Iniciando extracción de datos para: NZL...
Iniciando extracción de datos para: SCO...
Iniciando extracción robusta para: URU...
Iniciando extracción robusta para: AUS...
Iniciando extracción robusta para: BEL...
Iniciando extracción robusta para: IRQ...
Iniciando extracción robusta para: JPN...
Iniciando extracción robusta para: JOR...
Iniciando extracción robusta para: QAT...
Iniciando extracción robusta para: KSA...
Iniciando extracción robusta para: KOR...
Iniciando extracción robusta para: UZB...
Iniciando extracción robusta para: ALG...
Iniciando extracción robusta para: CPV...
Iniciando extracción robusta para: COD...
Iniciando extracción robusta para: EGY...
Iniciando extracción robusta para: GHA...
Iniciando extracción robus

In [7]:
df = df_games.reset_index(drop=True)

In [ ]:
df_lost = pd.read_excel('OtherGames.xlsx')
df_lost['Fecha'] = df_lost['Fecha'].dt.strftime("%Y-%m-%d")
df_final = pd.concat([df, df_lost], axis=0)
df_final.rename(columns={'Codigo_Pais':'Opponent', 'Seleccion':'Team', 
        'Condicion':'Condition', 'Goles_Equipo':'Goals_team', 
        'Goles_Rival':'Goals_opponent', 'Es_Penales':'Penalties', 
        'Es_Tiempo_Extra':'AET', 'Fecha':'Date',
        'Tipo_Partido_Limpio':'Game_type'}, inplace=True)

Save DataFrames

In [8]:
df.to_parquet('last_games.parquet', index=False)
df_ranking_fifa.to_parquet('ranking_fifa.parquet', index=False)
ranking_teams.to_parquet('teams.parquet', index=False)